# Null Imputation

Leakage-safe null imputation. Rules are learned from `train_split` only, then applied to `train_split`, `val_split`, and `test`. `health_condition` and `id` are never used as imputation predictors.

In [1]:
import itertools

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 160)

## Load Split Data

In [2]:
ID_COL = "id"
TARGET_COL = "health_condition"

train = pd.read_csv("data/train_split.csv")
val = pd.read_csv("data/val_split.csv")
test = pd.read_csv("data/test.csv")

print("train split shape:", train.shape)
print("validation split shape:", val.shape)
print("test shape:", test.shape)

train split shape: (552070, 15)
validation split shape: (138018, 15)
test shape: (295753, 14)


In [3]:
feature_cols = [col for col in train.columns if col not in [ID_COL, TARGET_COL]]
numeric_cols = train[feature_cols].select_dtypes(include="number").columns.tolist()
categorical_cols = train[feature_cols].select_dtypes(exclude="number").columns.tolist()

print("numeric feature columns:", numeric_cols)
print("categorical feature columns:", categorical_cols)

numeric feature columns: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
categorical feature columns: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


## Missing Value Summary

In [4]:
missing_summary = pd.DataFrame({
    "train_count": train[feature_cols].isna().sum(),
    "train_pct": train[feature_cols].isna().mean().mul(100),
    "val_count": val[feature_cols].isna().sum(),
    "val_pct": val[feature_cols].isna().mean().mul(100),
    "test_count": test[feature_cols].isna().sum(),
    "test_pct": test[feature_cols].isna().mean().mul(100),
    "dtype": train[feature_cols].dtypes.astype(str),
    "train_nunique": train[feature_cols].nunique(dropna=True),
}).sort_values("train_pct", ascending=False)

missing_summary.style.format({"train_pct": "{:.2f}", "val_pct": "{:.2f}", "test_pct": "{:.2f}"})

,train_count,train_pct,val_count,val_pct,test_count,test_pct,dtype,train_nunique
stress_level,66418,12.03,16393,11.88,35490,12.00,object,3
sleep_duration,60702,11.00,15297,11.08,32571,11.01,float64,699
sleep_quality,46733,8.47,11598,8.40,24999,8.45,object,3
calorie_expenditure,42122,7.63,10731,7.78,22652,7.66,float64,2092
water_intake,34865,6.32,8612,6.24,18633,6.30,float64,399
physical_activity_level,29253,5.30,7368,5.34,15695,5.31,object,3
smoking_alcohol,22920,4.15,5662,4.10,12249,4.14,object,3
gender,17100,3.10,4273,3.10,9160,3.10,object,3
step_count,11121,2.01,2795,2.03,5964,2.02,float64,12674
bmi,11086,2.01,2812,2.04,5956,2.01,float64,1585


## Train-Only Association Check

Association is used to decide where model-based imputation is worth using. Scores are learned from `train_split` only.

- Numeric vs numeric: absolute Spearman correlation
- Categorical vs categorical: bias-corrected Cramer's V
- Numeric vs categorical: correlation ratio

In [5]:
def _valid_pair(left, right):
    valid = left.notna() & right.notna()
    return left[valid], right[valid]


def is_numeric(series):
    return pd.api.types.is_numeric_dtype(series)


def cramers_v(left, right):
    left, right = _valid_pair(left, right)
    if left.nunique(dropna=True) < 2 or right.nunique(dropna=True) < 2:
        return np.nan

    observed = pd.crosstab(left, right)
    if observed.shape[0] < 2 or observed.shape[1] < 2:
        return np.nan

    observed_values = observed.to_numpy(dtype=float)
    n = observed_values.sum()
    row_sum = observed_values.sum(axis=1, keepdims=True)
    col_sum = observed_values.sum(axis=0, keepdims=True)
    expected = row_sum @ col_sum / n

    chi2 = np.divide(
        (observed_values - expected) ** 2,
        expected,
        out=np.zeros_like(expected),
        where=expected != 0,
    ).sum()

    phi2 = chi2 / n
    rows, cols = observed.shape
    phi2_corr = max(0, phi2 - ((cols - 1) * (rows - 1)) / (n - 1))
    rows_corr = rows - ((rows - 1) ** 2) / (n - 1)
    cols_corr = cols - ((cols - 1) ** 2) / (n - 1)
    denominator = min(cols_corr - 1, rows_corr - 1)

    if denominator <= 0:
        return np.nan
    return np.sqrt(phi2_corr / denominator)


def correlation_ratio(categories, values):
    categories, values = _valid_pair(categories, pd.to_numeric(values, errors="coerce"))
    valid_values = values.notna()
    categories = categories[valid_values]
    values = values[valid_values].astype(float)

    if categories.nunique(dropna=True) < 2 or values.nunique(dropna=True) < 2:
        return np.nan

    grouped = values.groupby(categories)
    counts = grouped.count()
    means = grouped.mean()
    overall_mean = values.mean()
    between_group_ss = (counts * (means - overall_mean) ** 2).sum()
    total_ss = ((values - overall_mean) ** 2).sum()

    if total_ss == 0:
        return np.nan
    return np.sqrt(between_group_ss / total_ss)


def association_score(left, right):
    left_is_numeric = is_numeric(left)
    right_is_numeric = is_numeric(right)

    if left_is_numeric and right_is_numeric:
        left, right = _valid_pair(left, right)
        if left.nunique(dropna=True) < 2 or right.nunique(dropna=True) < 2:
            return np.nan
        return abs(left.corr(right, method="spearman"))

    if not left_is_numeric and not right_is_numeric:
        return cramers_v(left, right)

    if left_is_numeric:
        return correlation_ratio(right, left)
    return correlation_ratio(left, right)


def column_kind(series):
    return "numeric" if is_numeric(series) else "categorical"

In [6]:
dependency_rows = []

for col_1, col_2 in itertools.combinations(feature_cols, 2):
    score = association_score(train[col_1], train[col_2])
    dependency_rows.append({
        "column_1": col_1,
        "column_2": col_2,
        "type_1": column_kind(train[col_1]),
        "type_2": column_kind(train[col_2]),
        "association": score,
        "non_null_pairs": int((train[col_1].notna() & train[col_2].notna()).sum()),
    })

dependency_summary = (
    pd.DataFrame(dependency_rows)
    .sort_values("association", ascending=False, na_position="last")
    .reset_index(drop=True)
)

dependency_summary.head(50).style.format({"association": "{:.3f}"})

,column_1,column_2,type_1,type_2,association,non_null_pairs
0,step_count,physical_activity_level,numeric,categorical,0.665,512283
1,exercise_duration,physical_activity_level,numeric,categorical,0.658,517609
2,step_count,exercise_duration,numeric,numeric,0.441,535540
3,sleep_duration,sleep_quality,numeric,categorical,0.376,449715
4,calorie_expenditure,physical_activity_level,numeric,categorical,0.372,482904
5,calorie_expenditure,exercise_duration,numeric,numeric,0.370,504829
6,calorie_expenditure,step_count,numeric,numeric,0.367,499683
7,bmi,calorie_expenditure,numeric,numeric,0.110,499703
8,bmi,stress_level,numeric,categorical,0.084,475881
9,sleep_duration,stress_level,numeric,categorical,0.081,432166


In [7]:
association_matrix = pd.DataFrame(
    np.eye(len(feature_cols)),
    index=feature_cols,
    columns=feature_cols,
)

for row in dependency_summary.itertuples(index=False):
    association_matrix.loc[row.column_1, row.column_2] = row.association
    association_matrix.loc[row.column_2, row.column_1] = row.association

association_matrix.style.background_gradient(cmap="viridis", axis=None).format("{:.2f}")

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
sleep_duration,1.00,0.00,0.06,0.00,0.01,0.00,0.00,0.01,0.08,0.38,0.01,0.04,0.00
heart_rate,0.00,1.00,0.00,0.00,0.01,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.00
bmi,0.06,0.00,1.00,0.11,0.02,0.02,0.00,0.00,0.08,0.03,0.04,0.02,0.00
calorie_expenditure,0.00,0.00,0.11,1.00,0.37,0.37,0.00,0.01,0.00,0.00,0.37,0.01,0.00
step_count,0.01,0.01,0.02,0.37,1.00,0.44,0.00,0.01,0.02,0.00,0.66,0.01,0.00
exercise_duration,0.00,0.01,0.02,0.37,0.44,1.00,0.00,0.01,0.01,0.00,0.66,0.01,0.00
water_intake,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.01,0.01,0.00,0.01,0.00,0.01
diet_type,0.01,0.01,0.00,0.01,0.01,0.01,0.01,1.00,0.01,0.00,0.01,0.00,0.01
stress_level,0.08,0.01,0.08,0.00,0.02,0.01,0.01,0.01,1.00,0.02,0.02,0.04,0.01
sleep_quality,0.38,0.00,0.03,0.00,0.00,0.00,0.00,0.00,0.02,1.00,0.00,0.01,0.00


## Association-Based Imputation

First, train-fitted median/mode imputers fill all missing values as a fallback. Then, for target columns that have train-only association >= `0.40` with other features, a model is trained on known `train_split` rows and applied to missing rows in `train_split`, `val_split`, and `test`.

In [8]:
ASSOCIATION_THRESHOLD = 0.40
MIN_KNOWN_ROWS_FOR_MODEL = 1_000
MODEL_RANDOM_STATE = 42

train_imputed = train.copy()
val_imputed = val.copy()
test_imputed = test.copy()

numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

if numeric_cols:
    train_imputed[numeric_cols] = numeric_imputer.fit_transform(train[numeric_cols])
    val_imputed[numeric_cols] = numeric_imputer.transform(val[numeric_cols])
    test_imputed[numeric_cols] = numeric_imputer.transform(test[numeric_cols])

if categorical_cols:
    train_imputed[categorical_cols] = categorical_imputer.fit_transform(train[categorical_cols])
    val_imputed[categorical_cols] = categorical_imputer.transform(val[categorical_cols])
    test_imputed[categorical_cols] = categorical_imputer.transform(test[categorical_cols])


def get_strong_predictors(target_col, threshold=ASSOCIATION_THRESHOLD):
    predictors = []
    for predictor_col in feature_cols:
        if predictor_col == target_col:
            continue
        score = association_matrix.loc[target_col, predictor_col]
        if pd.notna(score) and score >= threshold:
            predictors.append((predictor_col, score))
    return sorted(predictors, key=lambda item: item[1], reverse=True)


def build_model_imputer(target_col, predictors):
    predictor_numeric_cols = [col for col in predictors if col in numeric_cols]
    predictor_categorical_cols = [col for col in predictors if col in categorical_cols]
    transformers = []

    if predictor_numeric_cols:
        transformers.append(("numeric", SimpleImputer(strategy="median"), predictor_numeric_cols))

    if predictor_categorical_cols:
        transformers.append((
            "categorical",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            predictor_categorical_cols,
        ))

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    estimator = (
        HistGradientBoostingRegressor(random_state=MODEL_RANDOM_STATE)
        if target_col in numeric_cols
        else HistGradientBoostingClassifier(random_state=MODEL_RANDOM_STATE)
    )

    return Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])


def predict_and_clip(model, target_col, frame):
    predictions = model.predict(frame)
    if target_col in numeric_cols:
        known_values = train[target_col].dropna()
        predictions = np.clip(predictions, known_values.min(), known_values.max())
    return predictions


def apply_model_imputation(frame_original, frame_imputed, target_col, predictor_cols, model):
    missing_mask = frame_original[target_col].isna()
    if missing_mask.any():
        frame_imputed.loc[missing_mask, target_col] = predict_and_clip(
            model,
            target_col,
            frame_imputed.loc[missing_mask, predictor_cols],
        )
    return int(missing_mask.sum())


imputation_targets = [
    col for col in feature_cols
    if train[col].isna().any() or val[col].isna().any() or test[col].isna().any()
]

imputation_plan_rows = []
model_reports = []

for target_col in imputation_targets:
    strong_predictors = get_strong_predictors(target_col)
    predictor_cols = [col for col, _ in strong_predictors]
    known_mask = train[target_col].notna()
    use_model = bool(predictor_cols) and known_mask.sum() >= MIN_KNOWN_ROWS_FOR_MODEL

    imputation_plan_rows.append({
        "target_column": target_col,
        "target_type": column_kind(train[target_col]),
        "train_missing": int(train[target_col].isna().sum()),
        "val_missing": int(val[target_col].isna().sum()),
        "test_missing": int(test[target_col].isna().sum()),
        "method": "model" if use_model else "median/mode fallback",
        "predictors_from_train_association": ", ".join(f"{col} ({score:.3f})" for col, score in strong_predictors),
    })

    if not use_model:
        continue

    model = build_model_imputer(target_col, predictor_cols)
    model.fit(train_imputed.loc[known_mask, predictor_cols], train.loc[known_mask, target_col])

    filled_train = apply_model_imputation(train, train_imputed, target_col, predictor_cols, model)
    filled_val = apply_model_imputation(val, val_imputed, target_col, predictor_cols, model)
    filled_test = apply_model_imputation(test, test_imputed, target_col, predictor_cols, model)

    model_reports.append({
        "target_column": target_col,
        "model_type": "regressor" if target_col in numeric_cols else "classifier",
        "predictors": predictor_cols,
        "known_train_rows": int(known_mask.sum()),
        "filled_train_rows": filled_train,
        "filled_val_rows": filled_val,
        "filled_test_rows": filled_test,
    })

imputation_plan = pd.DataFrame(imputation_plan_rows)
model_imputation_report = pd.DataFrame(model_reports)

print(f"Association threshold used for model imputation: {ASSOCIATION_THRESHOLD}")
display(imputation_plan)
print("Model imputers trained on train_split and applied to train_split/val_split/test:")
display(model_imputation_report)
print("Remaining missing values in train features:", int(train_imputed[feature_cols].isna().sum().sum()))
print("Remaining missing values in val features:", int(val_imputed[feature_cols].isna().sum().sum()))
print("Remaining missing values in test features:", int(test_imputed[feature_cols].isna().sum().sum()))

Association threshold used for model imputation: 0.4


,target_column,target_type,train_missing,val_missing,test_missing,method,predictors_from_train_association
0,sleep_duration,numeric,60702,15297,32571,median/mode fallback,
1,heart_rate,numeric,6196,1637,3357,median/mode fallback,
2,bmi,numeric,11086,2812,5956,median/mode fallback,
3,calorie_expenditure,numeric,42122,10731,22652,median/mode fallback,
4,step_count,numeric,11121,2795,5964,model,"physical_activity_level (0.665), exercise_duration (0.441)"
5,exercise_duration,numeric,5511,1390,2958,model,"physical_activity_level (0.658), step_count (0.441)"
6,water_intake,numeric,34865,8612,18633,median/mode fallback,
7,diet_type,categorical,5532,1369,2958,median/mode fallback,
8,stress_level,categorical,66418,16393,35490,median/mode fallback,
9,sleep_quality,categorical,46733,11598,24999,median/mode fallback,


Model imputers trained on train_split and applied to train_split/val_split/test:


,target_column,model_type,predictors,known_train_rows,filled_train_rows,filled_val_rows,filled_test_rows
0,step_count,regressor,"[physical_activity_level, exercise_duration]",540949,11121,2795,5964
1,exercise_duration,regressor,"[physical_activity_level, step_count]",546559,5511,1390,2958
2,physical_activity_level,classifier,"[step_count, exercise_duration]",522817,29253,7368,15695


Remaining missing values in train features: 0


Remaining missing values in val features: 0
Remaining missing values in test features: 0


## Save Imputed Splits

In [9]:
train_imputed_path = "data/train_split_imputed.csv"
val_imputed_path = "data/val_split_imputed.csv"
test_imputed_path = "data/test_imputed.csv"

train_imputed.to_csv(train_imputed_path, index=False)
val_imputed.to_csv(val_imputed_path, index=False)
test_imputed.to_csv(test_imputed_path, index=False)

print("saved:", train_imputed_path, train_imputed.shape)
print("saved:", val_imputed_path, val_imputed.shape)
print("saved:", test_imputed_path, test_imputed.shape)

saved: data/train_split_imputed.csv (552070, 15)
saved: data/val_split_imputed.csv (138018, 15)
saved: data/test_imputed.csv (295753, 14)


In [10]:
train_imputed.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,313415,at-risk,7.17,91.3,26.86,2635.0,1398.0,49.1,2.03,non-veg,medium,average,sedentary,yes,male
1,3515,at-risk,6.99,75.1,24.23,2382.0,13466.0,50.8,1.83,balanced,low,average,active,occasional,male
2,501194,at-risk,8.66,82.4,21.41,2314.0,9473.0,23.3,3.09,balanced,low,poor,sedentary,yes,female
3,303602,at-risk,6.99,74.5,22.59,2165.0,7052.0,21.6,1.92,veg,medium,average,sedentary,occasional,male
4,117943,at-risk,8.83,68.2,22.01,2108.0,13521.0,52.9,2.35,non-veg,medium,average,active,yes,male


In [11]:
val_imputed.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,304516,at-risk,6.82,67.9,27.17,2464.0,13456.0,39.9,2.05,veg,medium,good,moderate,no,other
1,165358,at-risk,7.97,92.9,16.86,2240.0,7456.0,26.9,2.14,balanced,high,good,moderate,no,female
2,671841,at-risk,6.99,75.1,22.18,2352.0,4140.0,19.7,2.33,non-veg,low,poor,sedentary,occasional,male
3,403460,at-risk,6.99,83.1,21.93,2620.0,11656.0,41.1,1.22,non-veg,medium,good,moderate,occasional,other
4,351133,at-risk,8.14,80.0,23.41,2442.0,4484.0,38.9,2.32,veg,medium,average,sedentary,no,other


In [12]:
test_imputed.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,male
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other
